# polars-usaddress vs. Python `usaddress`: throughput benchmark

Compares the native Polars plugin (`polars_usaddress.tag_address`, one vectorized
call over a whole column) against the reference implementation
(`usaddress.tag`, called once per address, as a typical Python caller would use it).

**Prerequisite:** build the plugin into *this kernel's* environment first --
this notebook does not do that for you:

```bash
maturin develop --release
```

(`--release` matters: debug builds are ~20x slower and would make this an unfair fight.)

In [1]:
# Optional, only used for the charts at the end -- the notebook degrades gracefully without it.
%pip install --quiet matplotlib

/Users/zackery/Code/polars-usaddress/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import time
import random

import polars as pl

try:
    import polars_usaddress as plua
except ImportError as exc:
    raise SystemExit(
        "polars_usaddress isn't importable in this kernel.\n"
        "Build it into this kernel's environment first:\n\n"
        "    maturin develop --release\n"
    ) from exc

import usaddress
from usaddress import RepeatedLabelError

print("polars version:              ", pl.__version__)
print("polars_usaddress pinned to:  ", plua.UPSTREAM_VERSION)

polars version:               1.44.2
polars_usaddress pinned to:   0.5.16


## Synthetic benchmark corpus

Reuses the same randomised generator as `tools/verify_model.py`, so it exercises the
same tokenizer/feature-encoder edge cases the parity suite already covers -- directionals,
abbreviations, fractions, punctuation tails, PO boxes -- not just clean street addresses.
Seeded, so the corpus is identical on every run.

In [3]:
random.seed(1234)

NUM = ["1", "12", "123", "1200", "10000", "123A", "1/2", "\u00bd", "170th", "(45)"]
STREETS = ["Main", "Oak", "Elm", "Caf\u00e9", "\u00d1andu", "St", "Ave.", "Blvd", "N", "SW", "O'Hare"]
TAIL = ["", ",", ".", ";", ")", ",,", ".\n"]
EXTRA = ["#", "&", "Apt 3B", "PO Box 12", "Suite 100", "and", "Chicago, IL 60601", ""]


def make_corpus(n):
    out = []
    for _ in range(n):
        parts = [random.choice(NUM)] if random.random() < 0.8 else []
        for _ in range(random.randint(1, 4)):
            parts.append(random.choice(STREETS) + random.choice(TAIL))
        if random.random() < 0.5:
            parts.append(random.choice(EXTRA))
        out.append(" ".join(parts))
    return out


CORPUS_SIZE = 1_000_000
corpus = make_corpus(CORPUS_SIZE)
print(f"generated {len(corpus)} synthetic addresses")
corpus[:5]

generated 1000000 synthetic addresses


["Oak Oak.\n SW O'Hare,, and",
 '1 St,,',
 '123 Elm,, #',
 "1200 O'Hare)",
 '12 Ñandu.\n']

## Correctness spot-check

Before trusting the timing numbers, confirm the two engines actually agree on a sample.
A `RepeatedLabelError` from upstream is expected to come back as an all-null row from the
plugin (see `tag_address`'s docstring), so that case is handled specially rather than
counted as a mismatch.

In [4]:
def python_tag(addr):
    try:
        tagged, _addr_type = usaddress.tag(addr)
        return dict(tagged)
    except RepeatedLabelError:
        return None


SAMPLE_N = 300
sample = corpus[:SAMPLE_N]

rust_rows = (
    pl.DataFrame({"address": sample})
    .with_columns(parsed=plua.tag_address("address"))
    .unnest("parsed")
    .to_dicts()
)

mismatches = []
for addr, row in zip(sample, rust_rows):
    want = python_tag(addr)
    got = {k: v for k, v in row.items() if k in plua.LABELS and v is not None}
    if want is None:
        if got:
            mismatches.append((addr, "expected all-null (RepeatedLabelError upstream)", got))
        continue
    if got != want:
        mismatches.append((addr, want, got))

print(f"checked {SAMPLE_N} addresses, {len(mismatches)} mismatches")
for addr, want, got in mismatches[:5]:
    print("\n ", repr(addr))
    print("  want:", want)
    print("  got :", got)

checked 300 addresses, 28 mismatches

  "½ Ñandu, O'Hare, Blvd)"
  want: {'AddressNumber': '½', 'StreetName': "Ñandu, O'Hare", 'StreetNamePostType': 'Blvd)'}
  got : {'StreetName': "Ñandu, O'Hare", 'StreetNamePostType': 'Blvd)'}

  '½ Ave..'
  want: {'StreetName': '½', 'StreetNamePostType': 'Ave..'}
  got : {'StreetNamePostType': 'Ave..'}

  '½ Ave.. Main Suite 100'
  want: {'AddressNumber': '½', 'StreetName': 'Ave.. Main', 'OccupancyType': 'Suite', 'OccupancyIdentifier': '100'}
  got : {'AddressNumber': 'Ave..', 'StreetName': 'Main', 'OccupancyType': 'Suite', 'OccupancyIdentifier': '100'}

  '½ Ave.,, Blvd, and'
  want: {'AddressNumber': '½', 'StreetName': 'Ave.', 'StreetNamePostType': 'Blvd', 'IntersectionSeparator': 'and'}
  got : {'LandmarkName': 'Ave.,, Blvd', 'PlaceName': 'and'}

  '½ N.\n'
  want: {'AddressNumber': '½', 'StreetNamePreDirectional': 'N.\n'}
  got : {'StreetNamePreDirectional': 'N.\n'}


## Throughput: one batch at a fixed size

`time_python` calls `usaddress.tag` in a loop, the way a Python caller normally would.
`time_polars` builds a one-column DataFrame and makes a single vectorized call --
the way the plugin is meant to be used. Both include only the tagging work itself;
corpus generation happened above.

In [5]:
def time_python(addresses):
    start = time.perf_counter()
    for addr in addresses:
        python_tag(addr)
    return time.perf_counter() - start


def time_polars(addresses):
    df = pl.DataFrame({"address": addresses})
    start = time.perf_counter()
    df.with_columns(parsed=plua.tag_address("address")).unnest("parsed")
    # df.with_columns(parsed=plua.tag_address_with_confidence("address")).unnest("parsed")
    return time.perf_counter() - start


BENCH_N = 20_000  # keep the pure-Python side finishing in a reasonable time
bench_subset = corpus[:BENCH_N]  # owned by this section -- don't reuse this name below

# Warm-up: make sure lazy one-time costs (CRF model load, JIT-ish caches) aren't
# what we end up timing.
python_tag(bench_subset[0])
time_polars(bench_subset[:10])

bench_py_time = time_python(bench_subset)
bench_rs_time = time_polars(bench_subset)

print(f"{'engine':<24}{'total s':>10}{'addrs/s':>14}")
print(f"{'python usaddress':<24}{bench_py_time:>10.2f}{BENCH_N / bench_py_time:>14,.0f}")
print(f"{'polars_usaddress':<24}{bench_rs_time:>10.2f}{BENCH_N / bench_rs_time:>14,.0f}")
print(f"\nspeedup: {bench_py_time / bench_rs_time:,.1f}x")

engine                     total s       addrs/s
python usaddress              0.65        30,887
polars_usaddress              0.05       441,160

speedup: 14.3x


## Scaling sweep

Same comparison across corpus sizes, to see how much of the gap is fixed per-call overhead
(FFI, Python-object marshalling) versus per-address work.

In [12]:
sizes = [n for n in [100, 500, 1_000, 5_000, 20_000, 50_000,
                     1_000_000
                     ] if n <= len(corpus)]

rows = []
for n in sizes:
    sweep_subset = corpus[:n]
    py_t = time_python(sweep_subset)
    rs_t = time_polars(sweep_subset)
    rows.append({
        "n": n,
        "python_s": py_t,
        "polars_s": rs_t,
        "python_addrs_per_s": n / py_t,
        "polars_addrs_per_s": n / rs_t,
        "speedup": py_t / rs_t,
    })

results = pl.DataFrame(rows)
results

n,python_s,polars_s,python_addrs_per_s,polars_addrs_per_s,speedup
i64,f64,f64,f64,f64,f64
100,0.004877,0.001545,20502.831975,64738.914281,3.15756
500,0.018573,0.002086,26920.919309,239693.192665,8.903604
1000,0.03239,0.002888,30873.328985,346200.450151,11.213577
5000,0.16082,0.011076,31090.74891,451411.224418,14.519149
20000,0.637386,0.042078,31378.158155,475308.224923,15.147741
50000,1.592014,0.099985,31406.761452,500072.715587,15.922454
1000000,32.085081,2.121963,31167.133855,471261.742394,15.120471


In [7]:
results.write_clipboard()

In [8]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(results["n"], results["python_addrs_per_s"], marker="o", label="python usaddress")
    axes[0].plot(results["n"], results["polars_addrs_per_s"], marker="o", label="polars_usaddress")
    axes[0].set_xlabel("addresses")
    axes[0].set_ylabel("addresses / second")
    axes[0].set_title("Throughput")
    axes[0].legend()

    axes[1].plot(results["n"], results["speedup"], marker="o", color="tab:green")
    axes[1].set_xlabel("addresses")
    axes[1].set_ylabel("speedup (x)")
    axes[1].set_title("polars_usaddress speedup over python usaddress")

    fig.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed -- see the `results` table above instead.")

matplotlib not installed -- see the `results` table above instead.


## Isolating the CRF tagging cost

The scaling sweep above holds at ~2x from N=500 up, instead of growing with N --
that rules out fixed per-call/FFI overhead as the explanation (if it were that, the
speedup would keep climbing as N grows and overhead gets amortized). So the gap is a
genuine, roughly-constant *per-address* cost difference.

`usaddress.tag` does two things: build a Python feature dict per token
(`tokens2features`), then hand that to `pycrfsuite`'s C++ tagger. Only the first part
is pure Python -- the actual CRF inference is already C++ on both sides of this
comparison (CRFsuite vs. `crfs`, a from-scratch Rust port of the same format).
If CRF tagging itself is most of `usaddress.tag`'s time, then the win available to
the Rust plugin is capped by how `crfs`'s inference compares to CRFsuite's, not by
skipping Python's feature-dict construction.

This isolates tagging-only time by pre-encoding every address's feature sequence
once (outside the timed region), then timing just `tagger.tag(seq)` in a loop --
same corpus, same `usaddress.tag` cell above, so it's directly comparable to `py_time`.

In [9]:
import pycrfsuite

tagger_only = pycrfsuite.Tagger()
tagger_only.open(usaddress.MODEL_PATH)


def encode(tokens):
    return pycrfsuite.ItemSequence(usaddress.tokens2features(tokens))


# Pre-encode once, outside the timed region -- we're isolating tagging cost only.
# Uses bench_subset/bench_py_time from the "Throughput" section above, not the
# scaling sweep's sweep_subset -- so re-run that cell first if you've restarted
# the kernel or want a fresh measurement.
encoded_seqs = [encode(usaddress.tokenize(a)) for a in bench_subset]
encoded_seqs = [seq for seq in encoded_seqs if len(seq) > 0]

start = time.perf_counter()
for seq in encoded_seqs:
    tagger_only.tag(seq)
crf_only_time = time.perf_counter() - start

feature_time = bench_py_time - crf_only_time
print(f"full usaddress.tag() loop:      {bench_py_time:>8.2f} s  (n={BENCH_N})")
print(f"pycrfsuite tagging only:        {crf_only_time:>8.2f} s  ({100 * crf_only_time / bench_py_time:5.1f}% of total)")
print(f"tokenize + feature extraction:  {feature_time:>8.2f} s  ({100 * feature_time / bench_py_time:5.1f}% of total, by subtraction)")
print()
print(f"polars_usaddress total:         {bench_rs_time:>8.2f} s  (for comparison -- does tokenize + features + tagging, all in Rust)")

full usaddress.tag() loop:          0.65 s  (n=20000)
pycrfsuite tagging only:            0.15 s  ( 23.8% of total)
tokenize + feature extraction:      0.49 s  ( 76.2% of total, by subtraction)

polars_usaddress total:             0.05 s  (for comparison -- does tokenize + features + tagging, all in Rust)


### Cross-checking against the Rust-side split

`examples/bench_split.rs` measures the same feature-extraction/tagging split
directly in Rust, via `cargo run --release --no-default-features --features
bench-timing --example bench_split`. By default it cycles the ~40 addresses in
`tests/fixtures.json` (several of which are longer, multi-clause
`RepeatedLabelError` cases) rather than this notebook's synthetic `corpus` --
so its raw totals aren't comparable to `py_time`/`rs_time` above. Export the
exact same `subset` used in the cells above so the comparison is apples-to-apples:

In [10]:
import json
from pathlib import Path

corpus_path = Path("bench_corpus.json").resolve()
corpus_path.write_text(json.dumps(bench_subset))
print(f"wrote {len(bench_subset)} addresses to {corpus_path}")
print()
print("then, from the project root:")
print(
    "cargo run --release --no-default-features --features bench-timing "
    f"--example bench_split -- {corpus_path}"
)

wrote 20000 addresses to /Users/zackery/Code/polars-usaddress/tools/bench_corpus.json

then, from the project root:
cargo run --release --no-default-features --features bench-timing --example bench_split -- /Users/zackery/Code/polars-usaddress/tools/bench_corpus.json


## Python multiprocessing

`pycrfsuite` is a C extension and doesn't release the GIL during tagging, so Python
*threads* can't use extra cores here -- but separate *processes* can. The worker
function lives in `mp_worker.py`, not a cell in this notebook: `ProcessPoolExecutor`'s
default "spawn" start method (macOS/Windows) pickles a *reference* to the function and
re-imports it in each worker, and a function defined in a Jupyter cell lives in the
kernel's `__main__`, which a spawned worker can't re-import.

Each worker imports `usaddress` once when it starts (which is when `usaddress` loads its
own `pycrfsuite.Tagger`) and is then reused for many tasks -- the same "one per worker,
not one per row" shape as the Rust side's `map_init`. The pool is created once and kept
open across the warm-up and the timed run, so process-spawn/import cost lands in the
warm-up, not the measurement -- same reasoning as every other warm-up call above.

In [11]:
import os
import sys
from concurrent.futures import ProcessPoolExecutor

# mp_worker.py sits next to this notebook; add its directory to sys.path in case
# the kernel's cwd isn't already there.
for candidate in (os.getcwd(), os.path.join(os.getcwd(), "tools")):
    if os.path.exists(os.path.join(candidate, "mp_worker.py")) and candidate not in sys.path:
        sys.path.insert(0, candidate)

import mp_worker

N_WORKERS = os.cpu_count()
print(f"workers available: {N_WORKERS}")

# A handful of chunks per worker, so one short/slow chunk doesn't stall the whole
# pool, without paying IPC overhead per single address.
chunksize = max(1, len(bench_subset) // (N_WORKERS * 4))

with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
    # Warm-up: absorb worker spawn + usaddress/pycrfsuite import+model-load cost
    # here, not in the timed region below.
    list(pool.map(mp_worker.tag_one, bench_subset[:200], chunksize=1))

    start = time.perf_counter()
    list(pool.map(mp_worker.tag_one, bench_subset, chunksize=chunksize))
    mp_time = time.perf_counter() - start

mp_label = f"python usaddress (mp, {N_WORKERS} workers)"
print(f"{'engine':<32}{'total s':>10}{'addrs/s':>14}")
print(f"{'python usaddress (1 core)':<32}{bench_py_time:>10.2f}{BENCH_N / bench_py_time:>14,.0f}")
print(f"{mp_label:<32}{mp_time:>10.2f}{BENCH_N / mp_time:>14,.0f}")
print(f"{'polars_usaddress':<32}{bench_rs_time:>10.2f}{BENCH_N / bench_rs_time:>14,.0f}")
print()
print(f"multiprocessing speedup over single-core python: {bench_py_time / mp_time:,.1f}x")
print(f"polars_usaddress speedup over multiprocessing python: {mp_time / bench_rs_time:,.1f}x")

workers available: 8
engine                             total s       addrs/s
python usaddress (1 core)             0.65        30,887
python usaddress (mp, 8 workers)      0.17       120,027
polars_usaddress                      0.05       441,160

multiprocessing speedup over single-core python: 3.9x
polars_usaddress speedup over multiprocessing python: 3.7x


## Reading the numbers

- The flat ~2x speedup from N=500 upward (rather than growing with N) means this is a
  real per-address compute-cost ratio, not FFI/call overhead being amortized away.
- The cell above splits `usaddress.tag`'s time into feature extraction (pure Python)
  vs. CRF tagging (`pycrfsuite`, already C++). If tagging dominates, `crfs` -- a
  from-scratch pure-Rust CRFsuite port -- is the thing standing between this plugin
  and a much bigger speedup, and closing that gap means optimizing or replacing
  `crfs`'s inference path, not this crate's tokenizer/feature code.
- This measures single-threaded wall time for both. Polars can also parallelize this
  expression across rows via its query engine (e.g. inside a lazy `scan_csv(...).select(...)`
  pipeline) for a further win this notebook doesn't measure -- pure Python usaddress has
  no equivalent free lunch.